# Notebook para código da parte streaming em kafka


In [64]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# INSTALAÇÃO
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

import subprocess, os, time

KAFKA_VERSION = "3.7.0"
SCALA_VERSION = "2.13"
KAFKA_DIR     = f"/opt/kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
KAFKA_URL     = f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz"
KAFKA_ARCHIVE = "/tmp/kafka.tgz"
BIN           = f"{KAFKA_DIR}/bin"

# -- 1. Java --
print("Verificando / instalando Java...")
if subprocess.run(["java", "-version"], capture_output=True).returncode != 0:
    os.system("apt-get install -y -q default-jre-headless 2>/dev/null")
print("Java OK")

# -- 2. Download + extracao --
start_sh = f"{BIN}/zookeeper-server-start.sh"

if os.path.isfile(start_sh):
    print(f"Kafka ja instalado em {KAFKA_DIR}")
else:
    print(f"Baixando Kafka {KAFKA_VERSION} (~113 MB)...")
    os.system(f"rm -rf {KAFKA_DIR} {KAFKA_ARCHIVE}")

    ret = os.system(f"wget -q --show-progress {KAFKA_URL} -O {KAFKA_ARCHIVE}")

    if ret != 0 or not os.path.isfile(KAFKA_ARCHIVE):
        raise RuntimeError("Download falhou. Verifique a conexao.")

    size_mb = os.path.getsize(KAFKA_ARCHIVE) / (1024 * 1024)
    print(f"Arquivo baixado: {size_mb:.1f} MB")
    if size_mb < 50:
        raise RuntimeError(f"Arquivo muito pequeno ({size_mb:.1f} MB) — download incompleto.")

    print("Extraindo...")
    ret = os.system(f"tar -xzf {KAFKA_ARCHIVE} -C /opt/")
    if ret != 0:
        raise RuntimeError("Falha ao extrair o arquivo.")

    if not os.path.isfile(start_sh):
        raise RuntimeError(f"Extracao concluida mas {start_sh} nao encontrado.")

    print(f"Kafka extraido com sucesso em {KAFKA_DIR}")

print(f"\nScripts disponiveis em {BIN}/:")
scripts = [f for f in os.listdir(BIN) if f.endswith('.sh')]
print("   " + ", ".join(sorted(scripts)[:8]) + "...")

Verificando / instalando Java...
Java OK
Kafka ja instalado em /opt/kafka_2.13-3.7.0

Scripts disponiveis em /opt/kafka_2.13-3.7.0/bin/:
   connect-distributed.sh, connect-mirror-maker.sh, connect-plugin-path.sh, connect-standalone.sh, kafka-acls.sh, kafka-broker-api-versions.sh, kafka-client-metrics.sh, kafka-cluster.sh...


In [65]:
!pip install kafka-python

In [66]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# SUBINDO ZOOKEEPER
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import subprocess, os, time

KAFKA_DIR = "/opt/kafka_2.13-3.7.0"
BIN       = f"{KAFKA_DIR}/bin"
ZK_LOG    = "/tmp/zookeeper.log"

print("Verificando netcat...")
if subprocess.run(["which", "nc"], capture_output=True).returncode != 0:
    os.system("apt-get install -y -q netcat-openbsd 2>/dev/null")
    print("   netcat instalado.")
else:
    print("   netcat já disponível.")

os.system("pkill -f zookeeper 2>/dev/null; sleep 1")

ZK_CMD = f"{BIN}/zookeeper-server-start.sh {KAFKA_DIR}/config/zookeeper.properties"
print("Iniciando ZooKeeper...")

zk_proc = subprocess.Popen(
    ZK_CMD.split(),
    stdout=open(ZK_LOG, 'w'),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid
)

print("   Aguardando porta 2181", end="")
ok = False
for _ in range(20):
    time.sleep(1)
    print(".", end="", flush=True)
    chk = subprocess.run(
        ["nc", "-z", "-w1", "localhost", "2181"],
        capture_output=True
    )
    if chk.returncode == 0:
        ok = True
        break

print()
if ok:
    print("✅ ZooKeeper rodando na porta 2181")
else:
    print("❌ ZooKeeper não respondeu em 20s. Últimas linhas do log:")
    os.system(f"tail -30 {ZK_LOG}")

ERROR:kafka.net.manager:Connection failed: KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.net.connection:<KafkaConnection node_id=0 broker_version=(3, 7) (disconnected)>: Connection lost: KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.cluster:Metadata refresh: failed KafkaConnectionError: 111 ECONNREFUSED


Verificando netcat...
   netcat já disponível.
Iniciando ZooKeeper...
   Aguardando porta 2181

.

.
✅ ZooKeeper rodando na porta 2181


In [67]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# INICIANDO BROOKER
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
KAFKA_LOG = "/tmp/kafka.log"
KAFKA_CMD = f"{BIN}/kafka-server-start.sh {KAFKA_DIR}/config/server.properties"

print("Iniciando Kafka Broker...")
kafka_proc = subprocess.Popen(
    KAFKA_CMD.split(),
    stdout=open(KAFKA_LOG, 'w'),
    stderr=subprocess.STDOUT
)

time.sleep(8)

result = subprocess.run(["nc", "-z", "localhost", "9092"], capture_output=True)
if result.returncode == 0:
    print("Kafka Broker rodando na porta 9092")
else:
    print("❌ Kafka não respondeu — verifique o log:")
    os.system(f"tail -30 {KAFKA_LOG}")

Iniciando Kafka Broker...


❌ Kafka não respondeu — verifique o log:


In [68]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s"
)

log = logging.getLogger(__name__)

In [69]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# CRIANDO TÓPICOS DO PROJETO
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

from kafka import KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

BOOTSTRAP = "localhost:9092"

admin = KafkaAdminClient(
    bootstrap_servers=BOOTSTRAP
)

topicos = [

    NewTopic(
        name="alfabetizacao-indicadores",
        num_partitions=3,
        replication_factor=1
    ),

    NewTopic(
        name="alfabetizacao-metas",
        num_partitions=1,
        replication_factor=1
    )

]

for topico in topicos:

    try:

        admin.create_topics([topico])

        print(f"✅ {topico.name} criado.")

    except TopicAlreadyExistsError:

        print(f"ℹ️ {topico.name} já existe.")

ERROR:kafka.net.manager:Connection failed: KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.net.connection:<KafkaConnection node_id=bootstrap-0 broker_version=unknown (disconnected)>: Connection lost: KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.net.manager:Connection failed: KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.net.connection:<KafkaConnection node_id=bootstrap-0 broker_version=unknown (disconnected)>: Connection lost: KafkaConnectionError: 111 ECONNREFUSED


✅ alfabetizacao-indicadores criado.
✅ alfabetizacao-metas criado.


In [70]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# CRIANDO PRODUCER
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
from kafka import KafkaProducer
import json

def criar_producer():

    producer = KafkaProducer(

        bootstrap_servers="localhost:9092",

        value_serializer=lambda v: json.dumps(v).encode("utf-8")

    )

    log.info("Producer criado.")

    return producer

In [71]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# GERANDO EVENTOS DE SIMULAÇÃO
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

import random
from datetime import datetime

def gerar_evento():

    return {

        "tipo_evento": random.choice([
            "nova_medicao",
            "atualizacao_indicador",
            "atualizacao_meta"
        ]),

        "ano": random.choice([
            2023,
            2024,
            2025
        ]),

        "sigla_uf": random.choice([
            "SP",
            "MG",
            "GO",
            "BA",
            "PR"
        ]),

        "rede": random.choice([
            "Municipal",
            "Estadual"
        ]),

        "taxa_alfabetizacao":

            round(
                random.uniform(40,90),
                2
            ),

        "timestamp":

            datetime.now().isoformat()

    }

In [72]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# ENVIANDO EVENTOS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

import time

def enviar_eventos(qtd=10):

    producer = criar_producer()

    for i in range(qtd):

        evento = gerar_evento()

        producer.send(

            "alfabetizacao-indicadores",

            evento

        )

        print(evento)

        time.sleep(1)

    producer.flush()

    producer.close()

    log.info("Eventos enviados.")

In [73]:
enviar_eventos(10)

/tmp/ipykernel_415/758817114.py:9: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producer = criar_producer()


{'tipo_evento': 'nova_medicao', 'ano': 2023, 'sigla_uf': 'SP', 'rede': 'Municipal', 'taxa_alfabetizacao': 54.81, 'timestamp': '2026-07-14T01:01:13.238515'}
{'tipo_evento': 'atualizacao_indicador', 'ano': 2025, 'sigla_uf': 'GO', 'rede': 'Municipal', 'taxa_alfabetizacao': 41.82, 'timestamp': '2026-07-14T01:01:14.456708'}


{'tipo_evento': 'nova_medicao', 'ano': 2024, 'sigla_uf': 'GO', 'rede': 'Municipal', 'taxa_alfabetizacao': 50.85, 'timestamp': '2026-07-14T01:01:15.459119'}
{'tipo_evento': 'nova_medicao', 'ano': 2024, 'sigla_uf': 'BA', 'rede': 'Estadual', 'taxa_alfabetizacao': 68.83, 'timestamp': '2026-07-14T01:01:16.459729'}
{'tipo_evento': 'atualizacao_meta', 'ano': 2024, 'sigla_uf': 'GO', 'rede': 'Estadual', 'taxa_alfabetizacao': 85.88, 'timestamp': '2026-07-14T01:01:17.460402'}
{'tipo_evento': 'atualizacao_meta', 'ano': 2024, 'sigla_uf': 'PR', 'rede': 'Estadual', 'taxa_alfabetizacao': 81.16, 'timestamp': '2026-07-14T01:01:18.461912'}
{'tipo_evento': 'atualizacao_indicador', 'ano': 2023, 'sigla_uf': 'SP', 'rede': 'Municipal', 'taxa_alfabetizacao': 41.21, 'timestamp': '2026-07-14T01:01:19.463569'}
{'tipo_evento': 'atualizacao_indicador', 'ano': 2025, 'sigla_uf': 'MG', 'rede': 'Estadual', 'taxa_alfabetizacao': 42.37, 'timestamp': '2026-07-14T01:01:20.464235'}
{'tipo_evento': 'atualizacao_meta', 'ano':

In [74]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# CRIANDO CONSUMER
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

from kafka import KafkaConsumer

def criar_consumer():

    consumer = KafkaConsumer(

        "alfabetizacao-indicadores",

        bootstrap_servers="localhost:9092",

        auto_offset_reset="earliest",

        enable_auto_commit=True,

        value_deserializer=lambda x: json.loads(
            x.decode("utf-8")
        )

    )

    log.info("Consumer criado.")

    return consumer

In [75]:
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# CONSUMINDO EVENTOS
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

def consumir_eventos(qtd=10):

    consumer = criar_consumer()

    eventos = []

    print("Aguardando eventos...\n")

    for i, mensagem in enumerate(consumer):

        evento = mensagem.value

        print(evento)

        eventos.append(evento)

        if i + 1 >= qtd:
            break

    consumer.close()

    log.info(f"{len(eventos)} eventos consumidos.")

    return eventos

In [81]:
import pandas as pd

def salvar_bronze_stream(eventos):

    df = pd.DataFrame(eventos)

    caminho = "bronze_stream.parquet"

    df.to_parquet(
        caminho,
        index=False
    )

    log.info(f"Bronze Streaming salva em {caminho}")

    return df

In [82]:
def executar_streaming():

    log.info("Iniciando pipeline Streaming")

    enviar_eventos(10)

    eventos = consumir_eventos(10)

    df_stream = salvar_bronze_stream(eventos)

    log.info("Pipeline Streaming concluído.")

    return df_stream

In [83]:
df_stream = executar_streaming()

display(df_stream.head())

/tmp/ipykernel_415/758817114.py:9: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producer = criar_producer()


{'tipo_evento': 'atualizacao_indicador', 'ano': 2023, 'sigla_uf': 'PR', 'rede': 'Estadual', 'taxa_alfabetizacao': 75.21, 'timestamp': '2026-07-14T01:06:45.031608'}
{'tipo_evento': 'atualizacao_meta', 'ano': 2025, 'sigla_uf': 'GO', 'rede': 'Municipal', 'taxa_alfabetizacao': 78.7, 'timestamp': '2026-07-14T01:06:46.133548'}
{'tipo_evento': 'nova_medicao', 'ano': 2025, 'sigla_uf': 'MG', 'rede': 'Municipal', 'taxa_alfabetizacao': 63.04, 'timestamp': '2026-07-14T01:06:47.134165'}
{'tipo_evento': 'atualizacao_meta', 'ano': 2024, 'sigla_uf': 'PR', 'rede': 'Municipal', 'taxa_alfabetizacao': 73.75, 'timestamp': '2026-07-14T01:06:48.135131'}
{'tipo_evento': 'atualizacao_indicador', 'ano': 2024, 'sigla_uf': 'GO', 'rede': 'Municipal', 'taxa_alfabetizacao': 76.33, 'timestamp': '2026-07-14T01:06:49.135762'}
{'tipo_evento': 'atualizacao_indicador', 'ano': 2023, 'sigla_uf': 'BA', 'rede': 'Municipal', 'taxa_alfabetizacao': 43.57, 'timestamp': '2026-07-14T01:06:50.137224'}
{'tipo_evento': 'atualizacao_me

/tmp/ipykernel_415/995474878.py:9: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Aguardando eventos...

{'tipo_evento': 'atualizacao_meta', 'ano': 2024, 'sigla_uf': 'PR', 'rede': 'Estadual', 'taxa_alfabetizacao': 81.16, 'timestamp': '2026-07-14T01:01:18.461912'}
{'tipo_evento': 'atualizacao_indicador', 'ano': 2023, 'sigla_uf': 'SP', 'rede': 'Municipal', 'taxa_alfabetizacao': 41.21, 'timestamp': '2026-07-14T01:01:19.463569'}
{'tipo_evento': 'atualizacao_meta', 'ano': 2023, 'sigla_uf': 'MG', 'rede': 'Municipal', 'taxa_alfabetizacao': 89.18, 'timestamp': '2026-07-14T01:01:21.464874'}
{'tipo_evento': 'nova_medicao', 'ano': 2023, 'sigla_uf': 'SP', 'rede': 'Municipal', 'taxa_alfabetizacao': 49.06, 'timestamp': '2026-07-14T01:01:25.626337'}
{'tipo_evento': 'atualizacao_meta', 'ano': 2024, 'sigla_uf': 'PR', 'rede': 'Municipal', 'taxa_alfabetizacao': 69.69, 'timestamp': '2026-07-14T01:06:22.842867'}
{'tipo_evento': 'atualizacao_meta', 'ano': 2023, 'sigla_uf': 'SP', 'rede': 'Estadual', 'taxa_alfabetizacao': 44.83, 'timestamp': '2026-07-14T01:06:24.957555'}
{'tipo_evento': 'a

,tipo_evento,ano,sigla_uf,rede,taxa_alfabetizacao,timestamp
0,atualizacao_meta,2024,PR,Estadual,81.16,2026-07-14T01:01:18.461912
1,atualizacao_indicador,2023,SP,Municipal,41.21,2026-07-14T01:01:19.463569
2,atualizacao_meta,2023,MG,Municipal,89.18,2026-07-14T01:01:21.464874
3,nova_medicao,2023,SP,Municipal,49.06,2026-07-14T01:01:25.626337
4,atualizacao_meta,2024,PR,Municipal,69.69,2026-07-14T01:06:22.842867
